## Notebook09

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
movie = (
    pl.read_csv(ub + "data/movies_50_years.csv")
    .select(
        c.year,
        c.title,
        c.mpa,
        c.runtime,
        c.gross,
        c.rating,
        c.rating_count,
        c.metacritic
    )
)

### Questions

This week's data is the hundred highest grossing films in the United States in
each year from 1970 to 2019, which makes 5,000 rows. Each row is one film. The
`gross` column is domestic box office in millions of dollars, not adjusted for
inflation, so the numbers rise over the fifty years for reasons that have nothing
to do with how many tickets were sold. The `rating` is the IMDb user score out of
ten and `rating_count` is how many people voted; `metacritic` is a critic score
out of a hundred; `mpa` is the rating label (G, PG, R, and so on) and `runtime`
is in minutes.

Last time the table was complete and every group had something in it. This one is
full of holes, which is the point: this notebook is mostly about counting, and the
interesting thing about a count is what it does not include. Several questions
below group by decade, which is not a column in the data. Build it with integer
division, as in `(c.year // 10) * 10`.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Add a decade column and count the films in each decade with `pl.len()`.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(n = pl.len())
    .sort(c.decade)
)

A thousand films in each decade, which is what a hundred films a year for ten
years should give. Nothing is missing.

2. Now count the same groups a different way. Alongside `pl.len()`, use `.count()`
on the `metacritic` column and on `rating`. Then say what the three numbers are
counting.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(
        n = pl.len(),
        n_rating = c.rating.count(),
        n_metacritic = c.metacritic.count()
    )
    .sort(c.decade)
)

**Answer**:


3. That gap changes how you read an average. Compute the mean Metacritic score by
decade with the count sitting next to it.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(
        mc_avg = c.metacritic.mean(),
        n_metacritic = c.metacritic.count()
    )
    .sort(c.decade)
)

Every decade lands within about two points of 50, which looks like a tidy finding:
critics have been consistent for fifty years. Before believing it, check whether
the 73 films that carry a 1970s score resemble the 927 that do not. Group the
1970s films by whether the score is missing and compare them.

In [ ]:
(
    movie
    .filter(c.year < 1980)
    .with_columns(no_score = c.metacritic.is_null())
    .group_by(c.no_score)
    .agg(
        n = pl.len(),
        rating_avg = c.rating.mean(),
        gross_avg = c.gross.mean()
    )
)

**Answer**:


4. `.n_unique()` counts distinct values rather than rows, which is how you find
out whether a column identifies a row. Count the films in each decade and the
distinct titles in each decade side by side.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(
        n = pl.len(),
        n_titles = c.title.n_unique()
    )
    .sort(c.decade)
)

The 1970s have 1,000 films and 997 titles, so three titles appear twice. Titles
are not unique, and a film is identified by its title and year together, not by
its title. Across the whole table it happens 98 times: there are three films
called `Halloween` here, from 1978, 2007, and 2018.

5. `.agg()` belongs to a grouping, and there is no group here, so the obvious way
to summarize the whole table does not work. Try it and read the error.

In [ ]:
(
    movie
    .agg(n = pl.len())
)

**Answer**:


In [ ]:
(
    movie
    .select(
        n = pl.len(),
        n_titles = c.title.n_unique(),
        n_metacritic = c.metacritic.count(),
        rating_avg = c.rating.mean(),
        gross_total = c.gross.sum()
    )
)

One row, five columns. Fifty years of American cinema, or at least the profitable
end of it, comes to $264 billion.

6. The summary functions from the end of the chapter work the same way inside
`.agg()`. For each decade compute the smallest gross, the median, the ninetieth
percentile with `.quantile(0.9)`, and the largest.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(
        gross_min = c.gross.min(),
        gross_med = c.gross.median(),
        gross_q90 = c.gross.quantile(0.9),
        gross_max = c.gross.max()
    )
    .sort(c.decade)
)

Read the median column down and the box office grows by a factor of twelve, from
$5 million to $60 million, most of which is inflation and ticket prices rather
than audiences. Then read the minimum column, which is where the data stops making
sense. The smallest gross in the 1970s is 0.000005, which is five dollars. A film
that took five dollars is not among the hundred highest grossing films of its year,
so that number is not a small gross; it is a missing gross wearing a disguise. The
minimum and the maximum are the cheapest data check you own, and they are worth
running on any column before you trust anything else about it.

7. `pl.corr()` takes two columns instead of one and measures how tightly they move
together, on a scale from -1 to 1. Do critics and audiences agree? Correlate
`rating` with `metacritic` for the whole table, and then again within each decade.

In [ ]:
(
    movie
    .select(cor = pl.corr(c.rating, c.metacritic))
)

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(cor = pl.corr(c.rating, c.metacritic))
    .sort(c.decade)
)

About 0.36, and remarkably steady from decade to decade. Critics and audiences
point in the same direction and not much more than that; knowing that reviewers
liked a film tells you rather little about whether people did. Remember what this
number is computed from: only the 1,536 films that have both scores, which by
question 2 is a particular kind of film.

8. Now a question that aggregation cannot answer. Which film was the highest
grossing of each decade? Compute the largest gross per decade.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .group_by(c.decade, maintain_order=True)
    .agg(gross_max = c.gross.max())
    .sort(c.decade)
)

**Answer**:


9. Meanwhile there is a way to keep a title, though it takes some care. Sort the
films by gross, largest first, and use `.first()` inside `.agg()` to grab the title
at the top of each decade. Keep `.max()` next to it. Look hard at the answer before
you accept it.

In [ ]:
(
    movie
    .with_columns(decade = (c.year // 10) * 10)
    .sort(c.gross, descending=True)
    .group_by(c.decade, maintain_order=True)
    .agg(
        top = c.title.first(),
        gross_max = c.gross.max()
    )
    .sort(c.decade)
)

**Answer**:


10. Fix it. Drop the films with no recorded gross before sorting, and run the same
chain again.

In [ ]:
(
    movie
    .drop_nulls(c.gross)
    .with_columns(decade = (c.year // 10) * 10)
    .sort(c.gross, descending=True)
    .group_by(c.decade, maintain_order=True)
    .agg(
        top = c.title.first(),
        gross_max = c.gross.max()
    )
    .sort(c.decade)
)

Star Wars, E.T., Titanic, Avatar, and The Force Awakens. Now the two columns agree,
and so does everything you already knew about movies before you opened the file.
That last part is not a small thing. The reason we caught the bug in question 9 is
that the answer was famous enough to be obviously wrong, and most wrong answers are
not so considerate.